# 05 Lookup tables: CYP3A4 substrates and QT drugs

Build the drug side of the screen from published sources rather than from predictions. Condition 2 comes from the FDA table of CYP3A4 substrates, and condition 3 comes from the CredibleMeds QT list.

Intersecting the two gives the drugs that are both cleared by CYP3A4 and known to affect QT, which is the set where a CYP3A4 inhibitor could plausibly matter.

## Two files have to be downloaded by hand first

Neither source can be fetched automatically, so this notebook reads them from `data/raw/` and tells you clearly if they are missing.

| File to create | Source | Access |
| --- | --- | --- |
| `data/raw/fda_cyp3a4_substrates.csv` | [FDA drug interactions table](https://www.fda.gov/drugs/drug-interactions-labeling/drug-development-and-drug-interactions-table-substrates-inhibitors-and-inducers) | public, no login |
| `data/raw/credible_meds_qt.csv` | [CredibleMeds QTdrugs list](https://crediblemeds.org/druglist) | **free registration required** |

Required columns:

- `fda_cyp3a4_substrates.csv` -> `name`, `strength` (`sensitive` or `moderate`, the FDA's own split)
- `credible_meds_qt.csv` -> `name`, `risk_category` (`known risk of TdP`, `possible risk`, `conditional risk`)

Run the cell below to write blank templates with the right headers if you have not made the files yet.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..", "src")))

import pandas as pd

import names
import combine

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")

FDA_PATH = os.path.join(RAW_DIR, "fda_cyp3a4_substrates.csv")
QT_PATH = os.path.join(RAW_DIR, "credible_meds_qt.csv")

In [ ]:
# Writes header-only templates for whichever file is missing. Existing files
# are never overwritten, so re-running this is safe.
TEMPLATES = {
    FDA_PATH: "name,strength\n",
    QT_PATH: "name,risk_category\n",
}

for path, header in TEMPLATES.items():
    if os.path.exists(path):
        print(f"present: {os.path.basename(path)} "
              f"({sum(1 for _ in open(path)) - 1} rows)")
    else:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write(header)
        print(f"CREATED EMPTY TEMPLATE: {path}")
        print(f"   -> fill it in before the rest of this notebook will do anything")

## Load both tables

In [ ]:
fda = pd.read_csv(FDA_PATH)
qt = pd.read_csv(QT_PATH)

print("FDA substrates:", fda.shape)
print("CredibleMeds QT:", qt.shape)

if len(fda) == 0 or len(qt) == 0:
    raise SystemExit(
        "One or both lookup tables are empty. Download them into data/raw/ "
        "as described at the top of this notebook, then re-run from here."
    )

# The FDA splits sensitive from moderate substrates. Keep that distinction:
# it decides how strong a claim a flagged pair can support.
print()
print("substrate strength breakdown:")
print(fda["strength"].value_counts().to_string())
print()
print("QT risk categories (kept as they come, never flattened into one yes):")
print(qt["risk_category"].value_counts().to_string())

## Normalise the names and report what fails to matchThe two sources disagree about spelling, salt forms and bracketed notes.`src/names.py` lowercases, drops bracketed text and strips trailing salt andhydrate words.The unmatched names are **printed, not discarded**. A silently dropped row herelooks exactly like "this drug is not a CYP3A4 substrate", which is the wronganswer, so anything in the lists below needs a human decision.

In [ ]:
report = names.match_report(fda["name"], qt["name"])

print(f"both lists:        {len(report['matched'])} drugs")
print(f"FDA only:          {len(report['left_only'])} drugs")
print(f"CredibleMeds only: {len(report['right_only'])} drugs")
print()
print("A drug in BOTH lists is CYP3A4 cleared and QT affecting: the scope of the screen.")
print()
print("first 40 FDA-only names (check these for spelling variants of QT drugs):")
for n in report["left_only"][:40]:
    print("   ", n)

## Intersect

In [ ]:
fda = fda.copy()
qt = qt.copy()
fda["key"] = fda["name"].map(names.normalize_drug_name)
qt["key"] = qt["name"].map(names.normalize_drug_name)

# Report duplicate keys before merging. Two rows with the same key would
# otherwise quietly multiply into extra pairs in the final screen.
for label, frame in (("FDA", fda), ("QT", qt)):
    dupes = frame[frame["key"].duplicated(keep=False) & (frame["key"] != "")]
    if len(dupes):
        print(f"{label} duplicate keys to resolve by hand:")
        print(dupes.sort_values("key")[["name", "key"]].to_string(index=False))
        print()

intersected = fda.merge(qt, on="key", how="inner", suffixes=("_fda", "_qt"))
intersected = intersected.drop_duplicates(subset="key").reset_index(drop=True)

print(f"intersected: {len(intersected)} drugs")
print()
if len(intersected) < 10:
    print("WARNING: fewer than 10 drugs. The plan expected a few dozen.")
    print("Suspect the name matching before concluding the biology is surprising.")
    print("Check the FDA-only list printed above for spelling variants.")
print(intersected[["name_fda", "strength", "name_qt", "risk_category"]].to_string(index=False))

## SMILES for the intersected drugsModel B needs a structure for any drug that is a CYP3A4 substrate but is **not**on the QT list. Drugs that are on the list never reach Model B, so a missingSMILES for those is harmless.Add a `smiles` column to `data/raw/drug_smiles.csv` with a `name` column tomatch. PubChem is the easiest public source.

In [ ]:
SMILES_PATH = os.path.join(RAW_DIR, "drug_smiles.csv")

if not os.path.exists(SMILES_PATH):
    template = fda[["name"]].copy()
    template["smiles"] = ""
    template.to_csv(SMILES_PATH, index=False)
    print(f"CREATED TEMPLATE with {len(template)} drug names: {SMILES_PATH}")
    print("   -> fill in the smiles column from PubChem")
    drug_smiles = template
else:
    drug_smiles = pd.read_csv(SMILES_PATH)
    filled = drug_smiles["smiles"].notna() & (drug_smiles["smiles"].astype(str) != "")
    print(f"drug_smiles.csv: {filled.sum()} of {len(drug_smiles)} rows have a SMILES")

## Save the two lookup tables for combine.py

In [ ]:
substrates_out = fda[["name", "strength", "key"]].copy()
qt_out = qt[["name", "risk_category", "key"]].copy()

substrates_out.to_csv(os.path.join(PROCESSED_DIR, "cyp3a4_substrates.csv"), index=False)
qt_out.to_csv(os.path.join(PROCESSED_DIR, "qt_drugs.csv"), index=False)
intersected.to_csv(os.path.join(PROCESSED_DIR, "screen_drugs.csv"), index=False)

# Round trip them through the lookup builders so a key collision or a blank
# name shows up here rather than in the middle of the screen.
substrate_lookup = combine.build_substrate_lookup(substrates_out)
qt_lookup = combine.build_qt_lookup(qt_out)
print(f"substrate lookup: {len(substrate_lookup)} keys")
print(f"QT lookup:        {len(qt_lookup)} keys")
print()
print("saved cyp3a4_substrates.csv, qt_drugs.csv and screen_drugs.csv")